In [1]:
import requests
import pandas as pd
import time
from datetime import datetime
from tqdm.auto import tqdm

/opt/miniconda3/envs/cenv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', 2000)
pd.set_option('display.max_rows', None)

In [3]:
def get_reddit_leads(subreddits, keywords, hours_limit=24):
    """
    Scrapes Reddit for keyword mentions within a specific timeframe.
    Returns a cleaned, sorted, and formatted Pandas DataFrame with FULL text.
    """
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/120.0.0.0'
    }
    
    current_time = time.time()
    leads = []
    
    # Lowercase keywords once for efficiency
    keywords = [k.lower() for k in keywords]

    for sub in tqdm(subreddits, desc="Scraping Subreddits"):
        url = f"https://www.reddit.com/r/{sub}/new.json?limit=100"
        
        try:
            response = requests.get(url, headers=headers)
            if response.status_code != 200:
                continue 
                
            posts = response.json().get('data', {}).get('children', [])
            
            for post in posts:
                d = post['data']
                
                # 1. Filter by Time
                post_time = d.get('created_utc', 0)
                hours_ago = (current_time - post_time) / 3600
                
                if hours_ago > hours_limit:
                    continue
                    
                # 2. Filter by Keywords
                title = d.get('title', '')
                text = d.get('selftext', '')
                clean_full_text = text.strip().replace('\n', '   ') #3x space
                combined_text = f"{title} {text}".lower()
                
                if any(k in combined_text for k in keywords):
                    # Convert UTC to a clean, readable string
                    readable_date = datetime.fromtimestamp(post_time).strftime('%Y-%m-%d %H:%M:%S')
                    
                    leads.append({
                        "Subreddit": d.get('subreddit'),
                        "Author": d.get('author'),
                        "Posted_At": readable_date,
                        "Hours_Ago": round(hours_ago, 1),
                        "Score": int(d.get('score', 0)),
                        "Comments": int(d.get('num_comments', 0)),
                        "Title": title.strip(),
                        "Full_Text": clean_full_text,
                        "Upvote_Ratio": float(d.get('upvote_ratio', 0.0)),
                        "Post_URL": f"https://reddit.com{d.get('permalink')}",
                        "Author_URL": f"https://reddit.com/user/{d.get('author')}",
                        "Domain": d.get('domain'),
                        "Is_Video": bool(d.get('is_video', False))
                    })
                    
        except Exception as e:
            pass 

    # 3. Create DataFrame and apply final polish
    df = pd.DataFrame(leads)
    
    if not df.empty:
        # Sort by newest leads first and reset the index cleanly
        df = df.sort_values(by="Hours_Ago", ascending=True).reset_index(drop=True)
        
    return df

In [ ]:
# list of subreddits
my_subs = [
    "smallbusiness", "startups", "restaurantowners", "barowners", "sweatystartup", 
    "Entrepreneur", "restaurateur", "CoffeeShop", "foodtrucks", "cafe",
    "GoogleMyBusiness", "LocalSEO", "landscaping", "plumbing", "HVAC", 
    "cleaning_business", "PestControl", "salons", "Dentistry", "massage"
]

# list of keywords (Pain points, desires, and platforms)
my_keys = [
    "bad review", "google review", "yelp", "1-star", "one star", 
    "fake review", "negative review", "review bombed", "unfair review", 
    "tripadvisor", "google maps", "trustpilot", "get more reviews", 
    "increase reviews", "improve rating", "boost rating", 
    "remove a review", "delete a review", "dispute a review", 
    "customer feedback", "qr code"
]

# Run it for the last 24 hours
df_leads = get_reddit_leads(subreddits=my_subs, keywords=my_keys, hours_limit=24)
df_leads.shape

Scraping Subreddits: 100%|██████████| 19/19 [00:27<00:00,  1.45s/it]


(5, 13)

In [5]:
df_leads.head()

,Subreddit,Author,Posted_At,Hours_Ago,Score,Comments,Title,Full_Text,Upvote_Ratio,Post_URL,Author_URL,Domain,Is_Video
0,smallbusiness,ia2ai_official,2026-03-19 23:32:49,1.7,0,5,3 years running a junk removal business solo. finally stopped drowning in admin. here's what changed,"been running a junk removal company in LA for about 3 years now. solo operation, just me and my truck. got to the point where I had 400+ past clients but couldn't keep track of who I'd talked to, who needed follow up, when to ask for reviews, any of it. was spending like $400/month on different tools — CRM here, scheduling there, none of them talked to each other. spent more time on my laptop than actually hauling junk. something had to give. long story short I started automating everything. built a system that handles follow ups, sends review requests after jobs, tracks every client interaction, even gives me a morning report with my schedule and inbox. went from 20+ hours a week of admin to maybe 2. biggest thing that changed: I set up automated rebooking. system goes through my client list, finds everyone I haven't talked to in 3+ weeks, drafts personalized texts based on our actual conversation history. I approve them, they go out. sent 39 rebook texts today alone. used to do maybe 5 a week when I remembered. other stuff that moved the needle: - automated google review requests after every job (went from 3 reviews to getting consistent 5 stars) - morning briefing every day with weather + emails + calendar so I know if I need to reschedule - client database that actually works — can sort by who I haven't talked to longest and hit them first spring is the busiest season for junk removal and for the first time I actually feel like I can handle the volume without burning out. anyone else running a service biz solo and dealing with the admin nightmare? curious what you guys have figured out",0.33,https://reddit.com/r/smallbusiness/comments/1ryeosm/3_years_running_a_junk_removal_business_solo/,https://reddit.com/user/ia2ai_official,self.smallbusiness,False
1,localseo,Due-Bet115,2026-03-19 21:08:45,4.1,2,3,"Local SEO audit on 116k chiropractors: 42% no title, 67% no meta. Normal?","Hi everyone. Been pulling data from Google Maps listings and ran a full dataset of 116,426 chiropractors in the US. The pattern was consistent across the whole dataset. 67% have no meta description. 42% have no SEO page title. Most of these practices show up in organic search with random snippets and no positioning whatsoever. A few other things that stood out: \- Only 22% link to Instagram \- Only 56% have a website at all \- Only 37% have a Facebook presence High-trust category. High search intent before booking. But the on-site basics are almost universally missing. The GMB side looks more maintained than the website side. That split is what stood out most in the data. Have a good one.",1.00,https://reddit.com/r/localseo/comments/1ryaw70/local_seo_audit_on_116k_chiropractors_42_no_title/,https://reddit.com/user/Due-Bet115,self.localseo,False
2,GoogleMyBusiness,Mtru6,2026-03-19 19:16:07,5.9,1,7,My business profile page won't update,"When I google my business, the information in the sidebar result is incorrect. Screenshot [here](https://imgur.com/a/ixQ9Lkx). A bit of context: I am physical therapist doing home visits, and I guess this technically makes me a service area based business. When I first created the business profile the information provided was correct and it was nice to be discoverable on google maps despite not having a true brick and mortar storefront. Since then I have moved out of state and when I updated my business profile to change my location I was delisted and asked to provide verification. I updated the correct information and in my appeal I selected that I am a service area business. I won the appeal and business profile is listed again, but it still shows the old address. If you click on edit profile/edit business information and click location, it shows

In [6]:
df_leads[["Title", "Full_Text"]]

,Title,Full_Text
0,3 years running a junk removal business solo. finally stopped drowning in admin. here's what changed,"been running a junk removal company in LA for about 3 years now. solo operation, just me and my truck. got to the point where I had 400+ past clients but couldn't keep track of who I'd talked to, who needed follow up, when to ask for reviews, any of it. was spending like $400/month on different tools — CRM here, scheduling there, none of them talked to each other. spent more time on my laptop than actually hauling junk. something had to give. long story short I started automating everything. built a system that handles follow ups, sends review requests after jobs, tracks every client interaction, even gives me a morning report with my schedule and inbox. went from 20+ hours a week of admin to maybe 2. biggest thing that changed: I set up automated rebooking. system goes through my client list, finds everyone I haven't talked to in 3+ weeks, drafts personalized texts based on our actual conversation history. I approve them, they go out. sent 39 rebook texts today alone. used to do maybe 5 a week when I remembered. other stuff that moved the needle: - automated google review requests after every job (went from 3 reviews to getting consistent 5 stars) - morning briefing every day with weather + emails + calendar so I know if I need to reschedule - client database that actually works — can sort by who I haven't talked to longest and hit them first spring is the busiest season for junk removal and for the first time I actually feel like I can handle the volume without burning out. anyone else running a service biz solo and dealing with the admin nightmare? curious what you guys have figured out"
1,"Local SEO audit on 116k chiropractors: 42% no title, 67% no meta. Normal?","Hi everyone. Been pulling data from Google Maps listings and ran a full dataset of 116,426 chiropractors in the US. The pattern was consistent across the whole dataset. 67% have no meta description. 42% have no SEO page title. Most of these practices show up in organic search with random snippets and no positioning whatsoever. A few other things that stood out: \- Only 22% link to Instagram \- Only 56% have a website at all \- Only 37% have a Facebook presence High-trust category. High search intent before booking. But the on-site basics are almost universally missing. The GMB side looks more maintained than the website side. That split is what stood out most in the data. Have a good one."
2,My business profile page won't update,"When I google my business, the information in the sidebar result is incorrect. Screenshot [here](https://imgur.com/a/ixQ9Lkx). A bit of context: I am physical therapist doing home visits, and I guess this technically makes me a service area based business. When I first created the business profile the information provided was correct and it was nice to be discoverable on google maps despite not having a true brick and mortar storefront. Since then I have moved out of state and when I updated my business profile to change my location I was delisted and asked to provide verification. I updated the correct information and in my appeal I selected that I am a service area business. I won the appeal and business profile is listed again, but it still shows the old address. If you click on edit profile/edit business information and click location, it shows my new city. How do I show the correct information when I google my business name?"
3,What services do landlords and apartment building owners want but can't get?,"I live near a Mid-Size Metro City in the Midwest, and my gosh the city is adding about 3,000 new rental units a year. It seems everywhere I turn there is a new complex going up. Even in new housing developments, there are mix use buildings going up. So with all the new apartment growth, there has to be a service all these owners/landlords could use but maybe don't have yet? What about task of owning an operating a

In [8]:
from sentence_transformers import SentenceTransformer, util

In [9]:
model = SentenceTransformer('all-MiniLM-L6-v2')

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9891.32it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [10]:
def rank_leads_with_nlp(df_leads, saas_description, min_score=0.3):
    """
    Takes a DataFrame of Reddit posts and scores them against your SaaS description.
    Returns a sorted DataFrame with the most relevant leads at the top.
    """
    if df_leads.empty:
        print("No leads to score.")
        return df_leads

    print("🧠 Running NLP Matchmaking...")
    
    # 1. Combine Title and Text so the AI has full context
    texts_to_score = (df_leads['Title'] + " " + df_leads['Full_Text']).tolist()
    
    # 2. Convert text into NLP embeddings (Math vectors)
    post_embeddings = model.encode(texts_to_score, convert_to_tensor=True)
    saas_embedding = model.encode(saas_description, convert_to_tensor=True)
    
    # 3. Calculate "Cosine Similarity" (How closely the topics match)
    cosine_scores = util.cos_sim(saas_embedding, post_embeddings)[0]
    
    # 4. Add the scores to your DataFrame as a percentage
    df_leads['Match_Score'] = (cosine_scores.cpu().numpy() * 100).round(1)
    
    # 5. Filter out low scores and sort by the best matches
    df_ranked = df_leads[df_leads['Match_Score'] >= (min_score * 100)].copy()
    df_ranked = df_ranked.sort_values(by='Match_Score', ascending=False).reset_index(drop=True)
    
    print(f"✅ NLP Filtering complete. Kept {len(df_ranked)} highly relevant leads.")
    return df_ranked

In [ ]:
my_saas_desc = """
One bad Google review can cost a local business thousands. QRDive is a smart QR code that catches unhappy customers and routes them to a private feedback form, while automatically pushing your happy customers directly to Google to leave a 5-star review.
"""

# 2. Run the Scraper (From our previous code)
df_raw_leads = get_reddit_leads(subreddits=my_subs, keywords=my_keys, hours_limit=48)

# 3. Run the NLP Matchmaker
df_final_leads = rank_leads_with_nlp(df_raw_leads, my_saas_desc, min_score=0.3)

# 4. Print the top matches
if not df_final_leads.empty:
    for index, row in df_final_leads.iterrows():
        print(f"🔥 MATCH SCORE: {row['Match_Score']}% | r/{row['Subreddit']}")
        print(f"TITLE: {row['Title']}")
        print(f"LINK: {row['Post_URL']}")
        print("-" * 60)

Scraping Subreddits: 100%|██████████| 19/19 [00:26<00:00,  1.41s/it]


🧠 Running NLP Matchmaking...
✅ NLP Filtering complete. Kept 6 highly relevant leads.
🔥 MATCH SCORE: 52.599998474121094% | r/localseo
TITLE: Looking for SEO expert advice on this?
LINK: https://reddit.com/r/localseo/comments/1rx1a48/looking_for_seo_expert_advice_on_this/
------------------------------------------------------------
🔥 MATCH SCORE: 45.400001525878906% | r/localseo
TITLE: Why are some verified SABs completely invisible on Google Maps?
LINK: https://reddit.com/r/localseo/comments/1rxe8tn/why_are_some_verified_sabs_completely_invisible/
------------------------------------------------------------
🔥 MATCH SCORE: 45.099998474121094% | r/localseo
TITLE: Should I edit reply to old 1-star reviews? (new management)
LINK: https://reddit.com/r/localseo/comments/1rwpjj5/should_i_edit_reply_to_old_1star_reviews_new/
------------------------------------------------------------
🔥 MATCH SCORE: 42.400001525878906% | r/localseo
TITLE: Google My Business Verification tip if you encounter "No